# MMELON Single-View Inference with K-Fold Models

Score molecules from a CSV with the fold models from
[MMELON_singleview_finetuning_cv.ipynb](MMELON_singleview_finetuning_cv.ipynb) (`fold_<i>/best_model.ckpt`): single-view
models (text, image or graph). The view is read from the checkpoints.
For multi-view (MMELON) fold models, use [MMELON_inference_cv.ipynb](MMELON_inference_cv.ipynb).
Each molecule is scored by every fold model, and the final `score` is their average (an ensemble).
For a single model from [MMELON_singleview_finetuning.ipynb](MMELON_singleview_finetuning.ipynb), use
[MMELON_singleview_inference.ipynb](MMELON_singleview_inference.ipynb).

**Environment:** run with the `biomed-multiview` kernel.

**Input:** a CSV or parquet file with a SMILES column. A 0/1 label column is optional; if present, metrics
are reported. Only the SMILES, label and `KEEP_COLS` columns are read.

**Metrics** (only with labels), for each fold model and for the ensemble
- **hit@K** (hit@200, hit@100, hit@50): rank molecules by score; hit@K is how many of the top K have label 1.
- AUROC, PR AUC, and precision, recall, specificity and accuracy at `THRESHOLD`.

**Output**, in `<CV run folder>/predictions/<PREDICTION_NAME>/`:
```
predictions.csv   # every input row (row_index, SMILES, label, KEEP_COLS), plus score_fold0, score_fold1, ...,
                  # score (mean over folds), pred and rank (1 = highest score)
metrics.csv       # one row per fold model plus the ensemble; only if the CSV has labels
config.json       # settings used
```

Every fold model scores every molecule, so this takes K times as long as a single model
(roughly 0.25 s per molecule per fold on CPU); use a GPU for large files.

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this prediction; outputs go to <CV run folder>/predictions/<PREDICTION_NAME>/
PREDICTION_NAME = "test"

# CV run folder from MMELON_singleview_finetuning_cv.ipynb (contains fold_0/, fold_1/, ...)
CV_RUN_DIR = Path("../output") / DATASET / "singleview-cv_image"
N_FOLDS = None   # None = use every fold_<i>/best_model.ckpt found in CV_RUN_DIR

# Molecules to score
DATA_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"
KEEP_COLS = []        # other input columns to copy into predictions.csv, e.g. ["ID"]

In [2]:
import os

BATCH_SIZE = 32
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes: all CPUs, up to 24
THRESHOLD = 0.5           # score above this -> pred = 1
HIT_KS = [200, 100, 50]   # hit@K = number of label-1 molecules among the K highest-scored

In [3]:
import json
import shutil

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())

CV_RUN_DIR = Path(CV_RUN_DIR).expanduser().resolve()
assert CV_RUN_DIR.is_dir(), f"CV run folder not found: {CV_RUN_DIR}"
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"

found = sorted(CV_RUN_DIR.glob("fold_*/best_model.ckpt"), key=lambda p: int(p.parent.name.split("_")[1]))
if N_FOLDS is None:
    N_FOLDS = len(found)
MODEL_PATHS = [CV_RUN_DIR / f"fold_{i}" / "best_model.ckpt" for i in range(N_FOLDS)]
missing = [str(p) for p in MODEL_PATHS if not p.exists()]
assert N_FOLDS > 0 and not missing, f"Fold models not found in {CV_RUN_DIR}: {missing or 'no fold_*/best_model.ckpt'}"

OUT_DIR = CV_RUN_DIR / "predictions" / PREDICTION_NAME
assert not (OUT_DIR.exists() and any(OUT_DIR.iterdir())), (
    f"{OUT_DIR} already exists. Choose a new PREDICTION_NAME or delete that folder."
)
PIPELINE_DIR = OUT_DIR / "_pipeline"  # temporary copy for the data pipeline, deleted at the end
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

config = dict(cv_run_dir=str(CV_RUN_DIR), n_folds=N_FOLDS, models=[str(p) for p in MODEL_PATHS],
              data_path=str(DATA_PATH), smiles_col=SMILES_COL, label_col=LABEL_COL,
              threshold=THRESHOLD, hit_ks=HIT_KS)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Models : {N_FOLDS} folds in {CV_RUN_DIR}")
print(f"Data   : {DATA_PATH}")
print(f"Output : {OUT_DIR}")

Models : 1 folds in ~/repos/biomed-multi-view/aircheck/output/sample-data/singleview-cv_image
Data   : ~/repos/biomed-multi-view/aircheck/sample_data/test.parquet
Output : ~/repos/biomed-multi-view/aircheck/output/sample-data/singleview-cv_image/predictions/test


## 2. Prepare data

Every SMILES is checked with RDKit first: one unreadable SMILES would otherwise stop the whole run.
Invalid rows are kept in the output with an empty score. If the CSV has no label column,
predictions are still made and the metrics step is skipped.

In [4]:
import pandas as pd
from rdkit import Chem, RDLogger

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)

RDLogger.DisableLog("rdApp.*")

input_df = read_table(DATA_PATH, [SMILES_COL, *KEEP_COLS], optional=[LABEL_COL] if LABEL_COL else [])
input_df.insert(0, "row_index", range(len(input_df)))  # row in the input file
HAS_LABELS = LABEL_COL is not None and LABEL_COL in input_df.columns and input_df[LABEL_COL].notna().any()

smiles = input_df[SMILES_COL].astype(str).str.strip()
valid = input_df[SMILES_COL].notna() & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None)

# The MMELON pipeline reads data_predict.csv with columns smiles and label (a placeholder if unlabeled)
pipeline_df = pd.DataFrame({
    "smiles": smiles[valid],
    "label": input_df.loc[valid, LABEL_COL].fillna(0).astype(int) if HAS_LABELS else 0,
})
pipeline_df.to_csv(PIPELINE_DIR / "data_predict.csv", index=False)

print(f"Rows           : {len(input_df):,}")
print(f"Valid SMILES   : {valid.sum():,}")
if (~valid).any():
    print(f"Invalid SMILES : {(~valid).sum():,} (kept with an empty score), e.g. {input_df.loc[~valid, SMILES_COL].head(3).tolist()}")
print(f"Labels         : {'yes' if HAS_LABELS else 'no (metrics will be skipped)'}")

Rows           : 1,031
Valid SMILES   : 1,031
Labels         : yes


## 3. Load the data pipeline

The model's view (text, image or graph) is read from the checkpoint, and each SMILES is converted
into the input that view needs: text tokens, a 2D image or a molecular graph.

In [5]:
import gc
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bmfm_sm.core.data_modules.namespace import TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline

# Per view: model class and settings, data pipeline, input modality, and the prefix of that
# view's encoder weights inside the MMELON checkpoint
VIEW_SPECS = {
    "text": dict(
        model_class="bmfm_sm.predictive.modules.text_models.TextModel",
        model_params={},
        pipeline=TextFinetuneDataPipeline,
        modalities=["TEXT_MODEL"],
        weight_prefix="model_text.",
    ),
    "image": dict(
        model_class="bmfm_sm.predictive.modules.image_models.ImageModel",
        model_params={"baseModel": "ResNet18"},
        pipeline=ImageFinetuneDataPipeline,
        modalities=["IMAGE_MODEL"],
        weight_prefix="model_image.",
    ),
    "graph": dict(
        model_class="bmfm_sm.predictive.modules.graph_2d_models.Graph2dModel",
        model_params={"encoder_embed_dim": 512},
        pipeline=Graph2dFinetuneDataPipeline,
        modalities=["GRAPH_2D_MODEL"],
        weight_prefix="model_graph.",
    ),
}


def dataset_args_for(view: str) -> dict:
    return {
        "task_type": TaskType.CLASSIFICATION,
        "num_tasks": 1,
        "modalities": VIEW_SPECS[view]["modalities"],
        "smiles_col": "smiles",
        "label_cols": ["label"],
        "split_col": None,
    }


def view_of_checkpoint(path) -> str:
    # Which view a fine-tuned checkpoint was trained on, from the model class saved in it
    model_class = torch.load(path, map_location="cpu")["hyper_parameters"]["base_model_class"]
    matches = [v for v, s in VIEW_SPECS.items() if s["model_class"] == model_class]
    assert matches, f"{path} is not a single-view model ({model_class}); use the MMELON_* notebooks for multiview"
    return matches[0]


def to_device(batch):
    # Batches are dicts (multiview, graph) or tuples (text, image)
    if isinstance(batch, dict):
        return {k: to_device(v) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(v) for v in batch)
    return batch.to(DEVICE) if torch.is_tensor(batch) else batch

# The view (text, image or graph) is read from the checkpoints; all must be the same
views = {view_of_checkpoint(p) for p in MODEL_PATHS}
assert len(views) == 1, f"Models were trained on different views: {views}"
VIEW = views.pop()
config["view"] = VIEW
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

dataset = VIEW_SPECS[VIEW]["pipeline"](
    data_dir=str(PIPELINE_DIR), dataset_args=dataset_args_for(VIEW), stage="predict"
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


def predict(model_path: Path) -> np.ndarray:
    # Score every valid molecule (in data_predict.csv order) with one fine-tuned model
    model = FineTuneLightningModule.load_from_checkpoint(model_path, map_location=DEVICE)
    model.to(DEVICE).eval()
    scores = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=model_path.parent.name):
            batch = to_device(batch)
            logits = model(batch)  # encoder + prediction head (no fusion, so scores do not depend on the batch)
            scores.append(torch.sigmoid(logits).reshape(-1).cpu())
    del model
    gc.collect()
    scores = torch.cat(scores).numpy()
    assert len(scores) == len(pipeline_df), f"{len(scores)} scores for {len(pipeline_df)} molecules"
    return scores


def compute_metrics(labels, scores) -> dict:
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in HIT_KS}
    pred = scores > THRESHOLD
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels)) == 2
    metrics.update(
        rocauc=roc_auc_score(labels, scores) if both_classes else float("nan"),
        pr_auc=average_precision_score(labels, scores) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels),
    )
    return metrics


print(f"View: {VIEW}; device: {DEVICE}; {len(dataset):,} molecules in {len(loader):,} batches")

View: image; device: cuda; 1,031 molecules in 33 batches


## 4. Score the molecules with every fold model

In [6]:
%%time
predictions = input_df.copy()
fold_cols = []
for i, model_path in enumerate(MODEL_PATHS):
    col = f"score_fold{i}"
    predictions[col] = np.nan
    predictions.loc[valid, col] = predict(model_path)
    fold_cols.append(col)

predictions["score"] = predictions[fold_cols].mean(axis=1)  # ensemble: mean over folds
predictions["pred"] = (predictions["score"] > THRESHOLD).astype("Int64").where(valid)
predictions["rank"] = predictions["score"].rank(ascending=False, method="first").astype("Int64")
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)
shutil.rmtree(PIPELINE_DIR)

print(f"Saved {len(predictions):,} rows to {OUT_DIR / 'predictions.csv'}")
display(predictions.sort_values("rank").head(10))

2026-10-20 16:13:59,326 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Initializing model with params {'baseModel': 'ResNet18'}


2026-10-20 16:13:59,591 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Initialized the model with random weights as checkpoint_path was specified as None


2026-10-20 16:13:59,592 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Task type is classification


2026-10-20 16:13:59,600 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Biases in the last layer after initializing head weights: [-0.046828124672174454]


2026-10-20 16:13:59,601 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Base model weights frozen at start of training: False


...

2026-10-20 16:13:59,608 - root - INFO - instance-shay-ml-01:139887362635584:0:0 - Global seed: None


fold_0:   0%|          | 0/33 [00:00<?, ?it/s]

Saved 1,031 rows to ~/repos/biomed-multi-view/aircheck/output/sample-data/singleview-cv_image/predictions/test/predictions.csv


,row_index,SMILES,LABEL,score_fold0,score,pred,rank
332,332,Cc1cnc(C(=O)Nc2c(C)cccc2-c2nc3cc(F)ccc3n2C(C)C...,0,0.819383,0.819383,1,1
284,284,CN1C(=O)C(c2c(-c3ccc(Cl)cc3)[nH]c3c2c(=O)[nH]c...,1,0.818993,0.818993,1,2
144,144,CCN(C(=O)c1ccc(-n2nc(C)cc2C)cc1)c1ccccc1C,0,0.818990,0.818990,1,3
377,377,O=C(Nc1ccccc1)c1cc(-c2cccnc2)nc2ccccc12,1,0.818662,0.818662,1,4
399,399,O=C(Cc1noc2ccccc12)NC(c1ccc(F)cc1)C1CCCC1,0,0.818566,0.818566,1,5
275,275,O=C(NC1CC2(CCC2)Oc2ccccc21)c1ccc2nncn2c1,0,0.818533,0.818533,1,6
1027,1027,Cn1c(=O)c2cc(C(=O)N(CCc3cccs3)Cc3cccs3)cnc2n(C...,0,0.818514,0.818514,1,7
699,699,Cc1ccsc1CN(C)Cc1nc2scc(-c3ccccc3)c2c(=O)n1CC#N,0,0.818480,0.818480,1,8
708,708,Cc1ccc2cc(C(=O)NC(c3ccccc3)c3ccccn3)[nH]c2c1,1,0.818415,0.818415,1,9
44,44,Cc1ccc(C(CN2CCOCC2)NC(=O)c2ccc3ccccc3n2)cc1,0,0.818372,0.818372,1,10


CPU times: user 2.02 s, sys: 1.81 s, total: 3.83 s
Wall time: 1min 25s


## 5. Metrics (if the CSV has labels)

One row per fold model, plus `ensemble` (the mean score).

In [7]:
if not HAS_LABELS:
    print("No labels in the CSV; skipping metrics.")
else:
    scored = predictions[valid & predictions[LABEL_COL].notna()]
    rows = [dict(model=c.removeprefix("score_"), **compute_metrics(scored[LABEL_COL], scored[c]))
            for c in fold_cols + ["score"]]
    metrics = pd.DataFrame(rows).replace({"model": {"score": "ensemble"}})
    metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
    print(f"{len(scored):,} labeled molecules, {int(scored[LABEL_COL].sum()):,} positives")
    display(metrics.round(4))

1,031 labeled molecules, 31 positives


,model,hit@200,hit@100,hit@50,rocauc,pr_auc,precision,recall,specificity,accuracy
0,fold0,19,12,11,0.8105,0.1774,0.0351,1.0,0.147,0.1726
1,ensemble,19,12,11,0.8105,0.1774,0.0351,1.0,0.147,0.1726


In [8]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):40s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                                  0.00 MB
metrics.csv                                  0.00 MB
predictions.csv                              0.09 MB
